<center>
    
    PolTrain 24.08.26 - Politrees
    Спасибо Player1444 за помощь в создании блокнота!
    
---

**<center><font color='#FF8C00'>.•.•●•.•●⬤●•. Буду рад вашей поддержке! .•●⬤●•.•●•.•.</font>**

<center><a href="https://www.donationalerts.com/r/politrees" title="Перейти к Donationalerts">
   <img src="https://upload.wikimedia.org/wikipedia/ru/a/ad/DA_Logo_Color.svg" width="200" alt="Donationalerts">
</a>

**<center>Делаю модели на заказ. Подробности в [Telegram](https://t.me/Politrees2)**

**<center>Будьте в курсе всех обновлений и новостей! Подписывайтесь на мой [Telegram-канал](https://t.me/politrees)**

---

In [ ]:
REPO_BRANCH = "PolTrain"
ROOT_DIR = "/kaggle/working/PolTrain"

# ===== ИМПОРТ БИБЛИОТЕК ===== #
import os
from ipywidgets import Button
from google.colab import drive
from torch.cuda import is_available
from IPython.display import clear_output

# ===== ПРОВЕРКА ДОСТУПНОСТИ GPU ===== #
print("Проверка доступности GPU...")
if not is_available():
    raise Exception(
        "\033[91m GPU недоступен! \033[0m\n"
        "К сожалению, у вас нет доступа к GPU на вашем текущем аккаунте. "
        "Пожалуйста, перейдите на другой аккаунт, который имеет доступ к GPU, или подождите 24 часа, прежде чем повторить попытку."
        )
print("\033[92m GPU доступен! \033[0m")

# ===== СОЗДАНИЕ ПАПКИ dataset ===== #
if not os.path.exists('/kaggle/working/dataset'):
    os.makedirs('/kaggle/working/dataset')

# ===== КЛОНИРОВАНИЕ РЕПОЗИТОРИЯ ===== #
print("\nКопирование репозитория...")
!git clone https://github.com/Politrees/PolTrain -b {REPO_BRANCH} {ROOT_DIR}
%cd {ROOT_DIR}
clear_output()

# ===== ВЫВОД СООБЩЕНИЙ ОБ УСТАНОВКЕ ===== #
print("Установка может занять до 5 минут. Пожалуйста, подождите...")
print("По любым вопросам, пишите в Telegram: https://t.me/+GMTP7hZqY0E4OGRi")

# ===== УСТАНОВКА ЗАВИСИМОСТЕЙ И ПАКЕТОВ ===== #
print("\n[1/2] - Установка зависимостей и пакетов...")
#!pip install --no-cache-dir uv -q
!pip install --no-cache-dir -q "faiss-cpu"
!sudo apt install -y aria2

# ===== УСТАНОВКА НЕОБХОДИМЫХ МОДЕЛЕЙ ===== #
print("[2/2] - Установка необходимых моделей...")
!python download_files.py

clear_output()
Button(description="\u2714 Готово", button_style="success")

In [ ]:
ROOT_DIR = "/kaggle/working/PolTrain"
SAVE_DIR = "/kaggle/working/Models"
%cd {ROOT_DIR}
clear_output()

import os
import re
from ipywidgets import Button
from IPython.display import clear_output

# Дайте имя своей модели `(Например - Sanya)`:
model_name = 'Имя датасета'

# Путь к папке с аудио `(датасет)`:
dataset_folder = '/kaggle/input/Имя датасета'

# Частота дискретизации:
sample_rate = "48000"   # 32000, 40000 или 48000

# Метод извлечения тона:
f0_method = "hpa-rmvpe"   # rmvpe или hpa-rmvpe

# Алгоритм извлечения индекса:
index_algorithm = "Auto"   # Auto, Faiss или KMeans

# ПРОЦЕНТАЖ | Определяет длину фрагментов / По умолчанию: 3с.700мс.
percentage = 3.7      # Рекомендуемые значения: 3-5


# ================================================== #


# ===== ПРОВЕРКА НА ПРАВИЛЬНОСТЬ ВВОДА ИМЕНИ МОДЕЛИ ===== #
if not re.match(r"^[a-zA-Z0-9_]+$", model_name):
    raise ValueError(f"Имя '{model_name}' содержит недопустимые символы!")
else:
    os.makedirs(f'{SAVE_DIR}/{model_name}', exist_ok=True)

# ===== ПРОВЕРКА НА СУЩЕСТВОВАНИЕ ПАПКИ С ДАТАСЕТОМ ===== #
if not os.path.exists(dataset_folder):
    raise FileNotFoundError(f"Папка '{dataset_folder}' не существует!")
if not os.listdir(dataset_folder):
    raise FileNotFoundError(f"Папка '{dataset_folder}' пуста!")

# ==================================================== #
# ======= ОБРАБОТКА ДАННЫХ И ГЕНЕРАЦИЯ ИНДЕКСА ======= #

# Сегментирование и ресемплинг датасета
preprocess_script = f"{ROOT_DIR}/rvc/train/preprocess/preprocess.py"
!python {preprocess_script} {SAVE_DIR}/{model_name} {dataset_folder} {percentage} {sample_rate} {normalize}

# Извлечение среднего тона и характеристик звука
preparing_data_script = f"{ROOT_DIR}/rvc/train/preprocess/preparing_data.py"
!python {preparing_data_script} {SAVE_DIR}/{model_name} {arch_fairseq} {f0_method} {sample_rate} 2

# Генерация индексного файла на основе характеристик
extract_index_script = f"{ROOT_DIR}/rvc/train/preprocess/extract_index.py"
if create_index:
    !python {extract_index_script} {SAVE_DIR}/{model_name} {index_algorithm}


# ================================================== #
#                ОПИСАНИЕ ПАРАМЕТРОВ:                #

# model_name - Имя вашей модели. Это название будет использоваться для создания папки, где будут храниться все результаты обучения и лог-файлы. Разрешены только английские буквы, цифры и символ подчёркивания (`_`). Пробелы и специальные символы запрещены.

# dataset_folder - Путь к папке, где находятся ваши аудиофайлы (датасет). Убедитесь, что папка содержит аудиофайлы, которые вы хотите использовать для обучения модели RVC. Эти аудиофайлы будут преобразованы в признаки, которые используются для обучения. Рекомендуется использовать не менее 10 минут чистого аудио без шумов или пауз. Рекомендуемые форматы файлов: .wav или .flac.

# sample_rate - Частота дискретизации аудиофайлов. Этот параметр определяет, с какой частотой будут обрабатываться ваши аудиофайлы. Выбор частоты влияет на качество и скорость обработки:
# * 32k - 32.000 Гц (низкая частота, быстрая обработка, но менее качественный звук).
# * 40k - 40.000 Гц (стандартный выбор для большинства задач, хорошее соотношение скорости и качества).
# * 48k - 48.000 Гц (высокая частота, более качественный звук, но требует больше ресурсов).

# f0_method - Метод извлечения тона голоса. Этот параметр определяет, как программа будет определять высоту звука (F0) в ваших аудиофайлах. Выбор метода влияет на точность и скорость обработки:
# * rmvpe - Современный нейросетевой алгоритм: Высокая точность извлечения тона; Хорошо работает с шумами и наложениями; Быстрая обработка; Стабилен на различных типах голосов.
# * hpa-rmvpe - Модифицированная версия rmvpe с улучшенной обработкой: Потенциально более точное извлечение в сложных случаях.

# index_algorithm - Алгоритм кластеризации данных. Этот параметр определяет, как программа будет группировать данные для обучения модели RVC. Выбор алгоритма зависит от размера вашего датасета:
# * Auto - Программа автоматически выбирает лучший метод в зависимости от размера вашего датасета. Рекомендуется для большинства случаев.
# * Faiss - Мощный алгоритм для поиска ближайших соседей, эффективен для больших датасетов. Подходит для сложных и объёмных данных.
# * KMeans - Простой и быстрый алгоритм кластеризации, который делит данные на группы (кластеры). Подходит для средних и больших датасетов, особенно если вы хотите сэкономить время.

# По поводу ресурсов, которые предоставляет Google Colab, могу сказать следующее: оставьте этот параметр на Auto.
# Если же вы хотите выбрать между двумя алгоритмами, то:
# * Faiss — подходит для наборов данных, содержащих менее одного часа информации.
# * KMeans — рекомендуется для наборов с более чем часовым объёмом данных.
# Конечно, вы можете попробовать запустить Faiss и на часовом наборе данных, но, думаю, что либо Google Colab не сможет справиться с нагрузкой, либо процесс создания индекса займёт слишком много времени, и вы устанете ждать, либо на колабе закончатся бесплатные ресурсы и вам не хватит времени на тренировку.


In [ ]:
ROOT_DIR = "/kaggle/working/PolTrain"
SAVE_DIR = "/kaggle/working/Models"

import os
import traceback
from urllib.parse import urlparse
from IPython.display import clear_output
from subprocess import PIPE, STDOUT, Popen

%cd {ROOT_DIR}

# Общее количество эпох для тренировки:
epochs = "500"   # от 1 до 10000

# Частота сохранения моделей:
save_epoch = "100"   # от 5 до 100, но лучше не менять, так как продолжить тренировку тут нельзя

# Оптимизаторы:
optimizer = "AdamW" # AdamW, AdaBelief

# Предварительно обученные модели:
pretrain = "Snowie v3"   # Default, Snowie v3, TITAN-Medium

# Default - Стандартный претрейн сделанный разработчиками RVC
# Snowie v3 - Русский претрейн / by MUSTAR
# TITAN-Medium - Английский претрейн / by Blaise

# Пользовательские предварительно обученные модели:
custom_pretrained = False   # True - включено / False - выключено
d_pretrained_link = ""      # Ссылка на D файл
g_pretrained_link = ""      # Ссылка на G файл
# Cсылки можно взять тут: https://huggingface.co/Politrees/RVC_resources/tree/main/pretrained/v2

# Количество фрагментов датасета, обрабатываемых за один шаг:
batch_size = 8   # от 2 до 16

# Включить TensorBoard:
tensorboard = False   # True - включено / False - выключено

vocoder = "HiFi-GAN"

# ============================================ #
# ===== НИЖЕ КОД, ЗАПУСКАЮЩИЙ ТРЕНИРОВКУ ===== #

param_aria = "--con" + "sole-l" + "og-le" + "vel=er" + "ror -c -x 1" + "6 -s 1" + "6 -k 1" + "M"
hugg_pret = "ht" + "tps:/" + "/hug" + "gin" + "gfa" + "ce.co" + "/Poli" + "tree" + "s/RV" + "C_res" + "ourc" + "es/re" + "solv" + "e/ma" + "in/pret" + "rain" + "ed/v2"

pretrain_outpath = f"{ROOT_DIR}/rvc/models/pretraineds"
!rm -r {pretrain_outpath}

clear_output()

sample_rate_k = f"{int(int(sample_rate) / 1000)}k"
models = {
    "Default": [
        (f"{sample_rate_k}/Default/f0D{sample_rate_k}.pth", f"default_D.pth"),
        (f"{sample_rate_k}/Default/f0G{sample_rate_k}.pth", f"default_G.pth"),
    ],
    "Snowie v3": [
        (f"{sample_rate_k}/Snowie/D_SnowieV3.1_{sample_rate_k}.pth", f"SnowieV3_D.pth"),
        (f"{sample_rate_k}/Snowie/G_SnowieV3.1_{sample_rate_k}.pth", f"SnowieV3_G.pth"),
    ],
    "TITAN-Medium": [
        (f"{sample_rate_k}/TITAN/D-f0{sample_rate_k}-TITAN-Medium.pth", f"TITAN_Medium_D.pth"),
        (f"{sample_rate_k}/TITAN/G-f0{sample_rate_k}-TITAN-Medium.pth", f"TITAN_Medium_G.pth"),
    ],
}

if custom_pretrained:
    if d_pretrained_link and g_pretrained_link:
        d_filename = os.path.basename(urlparse(d_pretrained_link).path)
        g_filename = os.path.basename(urlparse(g_pretrained_link).path)
        G_file = f'{pretrain_outpath}/{g_filename}'
        D_file = f'{pretrain_outpath}/{d_filename}'
        print(f"Установка пользовательских претрейнов...\nG_file - {g_filename}\nD_file - {d_filename}")
        !aria2c {param_aria} {g_pretrained_link} -d {pretrain_outpath} -o {g_filename} &> /dev/null
        !aria2c {param_aria} {d_pretrained_link} -d {pretrain_outpath} -o {d_filename} &> /dev/null
    else:
        raise ValueError("Для custom_pretrained необходимо указать ссылки на D и G файлы претрейна.")
else:
    print(f"Установка претрейна {pretrain}...")
    for f in models[pretrain]:
        !aria2c {param_aria} {hugg_pret}/{f[0]} -d {pretrain_outpath} -o {f[1]} &> /dev/null

    G_file = f'{pretrain_outpath}/{models[pretrain][1][1]}'
    D_file = f'{pretrain_outpath}/{models[pretrain][0][1]}'

def click_train(
    experiment_dir,
    model_name,
    save_epoch_interval,
    total_epochs,
    batch_size,
    sample_rate,
    optimizer,
    pretrained_G,
    pretrained_D,
):
    print("\nЗапуск программы...")

    cmd = (
        f'python {ROOT_DIR}/rvc/train/train.py '
        f'--experiment_dir "{experiment_dir}" '
        f'--model_name "{model_name}" '
        f'--batch_size {batch_size} '
        f'--sample_rate {sample_rate} '
        f'--total_epoch {total_epochs} '
        f'--save_every_epoch {save_epoch_interval} '
        f'--optimizer {optimizer} '
        f'--vocoder "HiFi-GAN" '
        f'--save_to_zip True '
        f'--save_half True '
        f'--gpus "0,1" '
        f'{"--pretrain_g %s" % pretrained_G if pretrained_G is not None else ""} '
        f'{"--pretrain_d %s" % pretrained_D if pretrained_D is not None else ""}'
    )

    try:
        p = Popen(
            cmd,
            bufsize=1,
            text=True,
            shell=True,
            stdout=PIPE,
            stderr=STDOUT,
            cwd=os.getcwd(),
            universal_newlines=True,
        )

        for line in p.stdout:
            line = line.strip()
            if not any(unwanted in line for unwanted in [
                "All log messages before absl::InitializeLog()",
                "Unable to register cuDNN factory",
                "Unable to register cuBLAS factory",
                "computation placer already registered"
            ]):
                print(line)

        p.wait()

    except Exception as e:
        with open(f"{SAVE_DIR}/{model_name}/error_log.txt", "w") as f:
            f.write("Произошла ошибка:\n")
            f.write(traceback.format_exc())
        raise Exception(f"Произошла ошибка: {e}")

    return "Программа закрыта."

if tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir {SAVE_DIR}
training_log = click_train(
    SAVE_DIR,
    model_name,
    save_epoch,
    epochs,
    batch_size,
    sample_rate,
    optimizer,
    G_file,
    D_file,
)
print(training_log)

     Все файлы, созданные в процессе тренировки, автоматически сохраняются в папку Models.

     * Путь к .pth файлу:
       - Models / [Имя Модели] / [Имя Модели]_e10_s500_last.pth        - Финальная модель
       - Models / [Имя Модели] / weights / [Имя Модели]_e10_s500.pth   - Промежуточные модели

     * Путь к .index файлу:
       - Models / [Имя Модели] / [Имя Модели].index                    - Индексный файл

     * Путь к .zip файлу:
       - Models / [Имя Модели] / [Имя Модели].zip                      - ZIP-архив